# NeuroMark AI — Real TRIBE v2 Colab API with Brain Plot

This is the complete Colab notebook for:

```text
External frontend
→ ngrok URL
→ Colab FastAPI server
→ real TRIBE v2 inference
→ NeuroMark vector
→ Emotion Tree classification
→ CRM-style ranking
→ TRIBE brain-response plot
→ JSON response back to frontend
```

## Important

This notebook does **not** hard-code Hugging Face or ngrok tokens. It asks for them through hidden `getpass()` prompts.

If you previously pasted tokens into chat or a notebook, rotate them before using this seriously.

## Runtime setup

Before running:

1. Colab → **Runtime → Change runtime type**
2. Select **T4 GPU**
3. Run Cell 1
4. Run Cell 2 to restart
5. Continue from Cell 3


## Cell 1 — Install dependencies

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!apt-get update -qq
!apt-get install -y -qq ffmpeg

!pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!pip install -q huggingface_hub gdown pandas numpy matplotlib opencv-python-headless pillow jedi
!pip install -q fastapi uvicorn python-multipart nest_asyncio pyngrok requests
!pip install -q --force-reinstall --no-cache-dir numpy==2.2.6

print("Install complete. Run Cell 2 to restart the runtime.")

## Cell 2 — Restart runtime

In [ ]:
import os
os.kill(os.getpid(), 9)

## Cell 3 — Verify environment

In [ ]:
import sys
import numpy as np
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert np.__version__ == "2.2.6", "NumPy should be 2.2.6. Factory reset and rerun Cell 1 if broken."
print("Environment OK.")

## Cell 4 — Secure token input

In [ ]:
from getpass import getpass
import os

HF_TOKEN = getpass("Paste your Hugging Face READ token: ")
NGROK_TOKEN = getpass("Paste your ngrok authtoken: ")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
os.environ["NGROK_TOKEN"] = NGROK_TOKEN

print("Tokens loaded into runtime environment only.")

## Cell 5 — Hugging Face login

In [ ]:
from huggingface_hub import login, whoami

login(token=os.environ["HF_TOKEN"])
print(whoami())

## Cell 6 — Imports and folders

In [ ]:
from pathlib import Path
import os
import re
import io
import cv2
import json
import time
import shutil
import hashlib
import zipfile
import warnings
import threading
import base64
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import nest_asyncio
import uvicorn
import requests
import matplotlib.pyplot as plt

from fastapi import FastAPI, UploadFile, File, Form, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

from pyngrok import ngrok

from tribev2.demo_utils import TribeModel
from tribev2.plotting import PlotBrain

warnings.filterwarnings("ignore")
nest_asyncio.apply()

BASE_DIR = Path("/content/neuromark_research_api")
UPLOAD_DIR = BASE_DIR / "uploads"
OUTPUT_DIR = BASE_DIR / "outputs"
CACHE_FOLDER = BASE_DIR / "cache"

for p in [BASE_DIR, UPLOAD_DIR, OUTPUT_DIR, CACHE_FOLDER]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("UPLOAD_DIR:", UPLOAD_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## Cell 7 — Load actual TRIBE v2 model

In [ ]:
model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)

plotter = PlotBrain(mesh="fsaverage5")

print("Actual TRIBE v2 loaded.")

## Cell 8 — Asset helpers

In [ ]:
def safe_filename(name: str) -> str:
    name = Path(name).name
    name = re.sub(r"[^a-zA-Z0-9._-]+", "_", name)
    return name[:120] or "uploaded_asset"

def stable_asset_id(path: Path) -> str:
    h = hashlib.sha256()
    h.update(path.name.encode("utf-8"))
    with open(path, "rb") as f:
        h.update(f.read(1024 * 1024))
    return h.hexdigest()[:16]

def is_video(path: Path) -> bool:
    return path.suffix.lower() in [".mp4"]

def is_image(path: Path) -> bool:
    return path.suffix.lower() in [".jpg", ".jpeg", ".png"]

def image_to_static_mp4(
    image_path: Path,
    out_path: Path,
    seconds: int = 5,
    fps: int = 8,
    size=(224, 224),
) -> Path:
    img = Image.open(image_path).convert("RGB").resize(size)
    frame = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, size)

    for _ in range(seconds * fps):
        writer.write(frame)

    writer.release()
    return out_path

def make_short_clip(video_path: Path, seconds: int = 5) -> Path:
    out_path = video_path.with_name(f"{video_path.stem}_short_{seconds}s.mp4")
    cmd = f'ffmpeg -y -i "{video_path}" -t {seconds} -vf scale=224:224 -r 15 -c:v libx264 -preset ultrafast -c:a aac "{out_path}"'
    os.system(cmd)
    return out_path

def prepare_asset_for_tribe(path: Path, use_short_clip: bool = True, short_seconds: int = 5) -> Path:
    if is_video(path):
        if use_short_clip:
            return make_short_clip(path, seconds=short_seconds)
        return path

    if is_image(path):
        out_path = path.with_name(f"{path.stem}_static_clip.mp4")
        return image_to_static_mp4(path, out_path, seconds=short_seconds)

    raise ValueError(f"Unsupported file type: {path.suffix}")

## Cell 9 — Real TRIBE inference

In [ ]:
def run_real_tribe_on_asset(asset_path: Path, use_short_clip: bool = True, short_seconds: int = 5):
    """
    Runs actual TRIBE v2 on a video/image asset.

    Returns:
        preds: real TRIBE predictions, usually [timesteps, vertices]
        segments: TRIBE stimulus segments
        df_events: event dataframe
        tribe_input_path: exact MP4 passed into TRIBE
    """
    tribe_input_path = prepare_asset_for_tribe(
        asset_path,
        use_short_clip=use_short_clip,
        short_seconds=short_seconds,
    )

    print("Running TRIBE on:", tribe_input_path)

    df_events = model.get_events_dataframe(video_path=tribe_input_path)
    import torch
    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            preds, segments = model.predict(events=df_events)

    preds = np.asarray(preds, dtype=np.float32)

    if preds.ndim != 2:
        raise ValueError(f"Expected 2D TRIBE output, got shape {preds.shape}")

    # Ensure [timesteps, vertices]
    if preds.shape[0] > preds.shape[1]:
        preds = preds.T

    return preds, segments, df_events, tribe_input_path

## Cell 10 — TRIBE brain plot generation

In [ ]:
def make_tribe_brain_plot_base64(preds, segments, asset_id="asset", max_timesteps=15):
    """
    Generates TRIBE brain-response plot and returns:
        png_path, base64_png
    """
    n = min(max_timesteps, len(preds), len(segments))

    fig = plotter.plot_timesteps(
        preds[:n],
        segments=segments[:n],
        cmap="fire",
        norm_percentile=99,
        vmin=.6,
        alpha_cmap=(0, .2),
        show_stimuli=True,
    )

    png_path = OUTPUT_DIR / f"{asset_id}_tribe_brain_plot.png"

    fig.savefig(
        png_path,
        dpi=180,
        bbox_inches="tight",
        facecolor="white",
    )

    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=180,
        bbox_inches="tight",
        facecolor="white",
    )
    buf.seek(0)

    b64 = base64.b64encode(buf.read()).decode("utf-8")

    plt.close(fig)

    return png_path, b64

## Cell 11 — Real TRIBE neural dynamics

In [ ]:
def robust_scale_0_100(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = np.percentile(x, 5), np.percentile(x, 95)

    if abs(hi - lo) < eps:
        return np.zeros_like(x) + 50.0

    y = (x - lo) / (hi - lo + eps)
    return np.clip(y * 100.0, 0, 100)

def curve_stats(curve):
    curve = np.asarray(curve, dtype=np.float32)
    return {
        "mean": round(float(np.mean(curve)), 3),
        "peak": round(float(np.max(curve)), 3),
        "volatility": round(float(np.std(curve)), 3),
        "spike_index": int(np.argmax(curve)),
    }

def extract_tribe_neural_dynamics(preds):
    P = np.asarray(preds, dtype=np.float32)
    absP = np.abs(P)

    global_activity_raw = absP.mean(axis=1)
    peak_activity_raw = absP.max(axis=1)
    spatial_variance_raw = absP.var(axis=1)
    cortical_spread_raw = (absP > np.percentile(absP, 75)).mean(axis=1)

    if absP.shape[0] > 1:
        temporal_change_raw = np.zeros(absP.shape[0], dtype=np.float32)
        temporal_change_raw[1:] = np.mean(np.abs(absP[1:] - absP[:-1]), axis=1)
    else:
        temporal_change_raw = np.zeros(absP.shape[0], dtype=np.float32)

    if absP.shape[0] >= 3:
        sustained_raw = np.convolve(global_activity_raw, np.ones(3) / 3, mode="same")
    else:
        sustained_raw = global_activity_raw.copy()

    curves = {
        "global_activity": robust_scale_0_100(global_activity_raw),
        "peak_activity": robust_scale_0_100(peak_activity_raw),
        "spatial_variance": robust_scale_0_100(spatial_variance_raw),
        "cortical_spread": robust_scale_0_100(cortical_spread_raw),
        "temporal_change": robust_scale_0_100(temporal_change_raw),
        "sustained_response": robust_scale_0_100(sustained_raw),
    }

    stats = {name: curve_stats(curve) for name, curve in curves.items()}

    return curves, stats

## Cell 12 — NeuroMark vector synthesis

In [ ]:
def mean_curve(curves, name):
    return float(np.mean(curves[name]))

def neuromark_vector_from_real_tribe(curves):
    global_activity = mean_curve(curves, "global_activity")
    peak_activity = mean_curve(curves, "peak_activity")
    spatial_variance = mean_curve(curves, "spatial_variance")
    cortical_spread = mean_curve(curves, "cortical_spread")
    temporal_change = mean_curve(curves, "temporal_change")
    sustained = mean_curve(curves, "sustained_response")

    visual_attention = 0.40 * global_activity + 0.25 * peak_activity + 0.20 * cortical_spread + 0.15 * temporal_change
    cognitive_load = 0.45 * spatial_variance + 0.30 * temporal_change + 0.25 * cortical_spread
    engagement = 0.45 * peak_activity + 0.35 * sustained + 0.20 * global_activity
    memory_encoding = 0.50 * sustained + 0.30 * visual_attention + 0.20 * engagement
    motivation_reward = 0.55 * engagement + 0.30 * visual_attention + 0.15 * peak_activity

    emotional_warmth = 0.45 * (100 - cognitive_load) + 0.30 * memory_encoding + 0.25 * sustained
    trust_safety = 0.55 * emotional_warmth + 0.30 * (100 - cognitive_load) + 0.15 * memory_encoding
    urgency = 0.50 * temporal_change + 0.30 * peak_activity + 0.20 * visual_attention
    confusion_risk = 0.55 * cognitive_load + 0.30 * temporal_change + 0.15 * spatial_variance
    calm_aesthetic_style = 0.65 * (100 - cognitive_load) + 0.20 * emotional_warmth + 0.15 * sustained
    theme_fit = 0.30 * memory_encoding + 0.25 * visual_attention + 0.25 * trust_safety + 0.20 * motivation_reward

    raw = {
        "visual_attention": visual_attention,
        "emotional_warmth": emotional_warmth,
        "motivation_reward": motivation_reward,
        "cognitive_load": cognitive_load,
        "memory_encoding": memory_encoding,
        "trust_safety": trust_safety,
        "urgency": urgency,
        "confusion_risk": confusion_risk,
        "calm_aesthetic_style": calm_aesthetic_style,
        "theme_fit": theme_fit,
    }

    return {k: round(float(np.clip(v, 0, 100)), 2) for k, v in raw.items()}

## Cell 13 — Emotion Tree classification

In [ ]:
def emotion_tree_classify(v, curves, fps=1):
    attention = v["visual_attention"]
    warmth = v["emotional_warmth"]
    motivation = v["motivation_reward"]
    load = v["cognitive_load"]
    memory = v["memory_encoding"]
    trust = v["trust_safety"]
    urgency = v["urgency"]
    confusion = v["confusion_risk"]
    calm = v["calm_aesthetic_style"]

    layer_scores = {
        "Primal": round(0.45 * attention + 0.35 * urgency + 0.20 * motivation, 2),
        "Emotional": round(0.40 * warmth + 0.35 * trust + 0.25 * motivation, 2),
        "Cognitive": round(0.45 * load + 0.35 * confusion + 0.20 * memory, 2),
    }

    emotion_scores = {
        "Trust": 0.45 * trust + 0.35 * warmth + 0.20 * calm,
        "Urgency": 0.55 * urgency + 0.25 * attention + 0.20 * motivation,
        "Motivation": 0.45 * motivation + 0.30 * memory + 0.25 * attention,
        "Cognitive Overload": 0.55 * load + 0.30 * confusion + 0.15 * urgency,
        "Calm Focus": 0.45 * calm + 0.30 * memory + 0.25 * trust,
        "Apathy": 100 - (0.45 * attention + 0.35 * motivation + 0.20 * memory),
    }

    emotion_scores = {k: round(float(np.clip(vv, 0, 100)), 2) for k, vv in emotion_scores.items()}

    detected = max(emotion_scores, key=emotion_scores.get)
    sorted_scores = sorted(emotion_scores.values(), reverse=True)
    confidence_gap = round(float(sorted_scores[0] - sorted_scores[1]), 2)

    cognitive_curve = curves["spatial_variance"] * 0.45 + curves["temporal_change"] * 0.35 + curves["cortical_spread"] * 0.20
    spike_idx = int(np.argmax(cognitive_curve))
    chaos_spike_second = round(float(spike_idx / max(fps, 1)), 3)

    dominant_layer = max(layer_scores, key=layer_scores.get)

    if detected == "Cognitive Overload":
        explanation = "High variance/change suggests processing friction or overload."
    elif detected == "Trust":
        explanation = "Low overload with stronger warmth/safety indicators."
    elif detected == "Urgency":
        explanation = "Strong temporal change and attention suggests urgency/action pressure."
    elif detected == "Motivation":
        explanation = "Strong engagement and sustained response suggests motivational pull."
    elif detected == "Calm Focus":
        explanation = "Lower overload with sustained processing and trust/safety alignment."
    else:
        explanation = "Weaker attention/motivation signals compared with other categories."

    return {
        "detected_emotion": detected,
        "emotion_scores": emotion_scores,
        "dominant_layer": dominant_layer,
        "layer_scores": layer_scores,
        "classification_confidence_gap": confidence_gap,
        "chaos_spike_second": chaos_spike_second,
        "explanation": explanation,
    }

## Cell 14 — Mock CRM database

In [ ]:
FEATURE_NAMES = [
    "visual_attention",
    "emotional_warmth",
    "motivation_reward",
    "cognitive_load",
    "memory_encoding",
    "trust_safety",
    "urgency",
    "confusion_risk",
    "calm_aesthetic_style",
    "theme_fit",
]

historical_campaigns = pd.DataFrame([
    {
        "campaign_id": "hist_001",
        "campaign_name": "Student exam countdown reel",
        "best_segment": "Exam-focused students",
        "platform": "TikTok/Reels",
        "visual_attention": 82,
        "emotional_warmth": 55,
        "motivation_reward": 86,
        "cognitive_load": 42,
        "memory_encoding": 78,
        "trust_safety": 58,
        "urgency": 88,
        "confusion_risk": 35,
        "calm_aesthetic_style": 44,
        "theme_fit": 84,
        "watch_time_score": 84,
        "save_rate_score": 70,
        "click_rate_score": 78,
        "purchase_rate_score": 72,
        "platform_fit": 91,
    },
    {
        "campaign_id": "hist_002",
        "campaign_name": "Calm productivity SaaS explainer",
        "best_segment": "Young professionals",
        "platform": "LinkedIn/YouTube",
        "visual_attention": 64,
        "emotional_warmth": 76,
        "motivation_reward": 68,
        "cognitive_load": 28,
        "memory_encoding": 72,
        "trust_safety": 80,
        "urgency": 32,
        "confusion_risk": 20,
        "calm_aesthetic_style": 82,
        "theme_fit": 79,
        "watch_time_score": 72,
        "save_rate_score": 64,
        "click_rate_score": 60,
        "purchase_rate_score": 62,
        "platform_fit": 82,
    },
    {
        "campaign_id": "hist_003",
        "campaign_name": "High-energy discount ad",
        "best_segment": "Deal seekers",
        "platform": "Meta Ads",
        "visual_attention": 88,
        "emotional_warmth": 42,
        "motivation_reward": 80,
        "cognitive_load": 70,
        "memory_encoding": 69,
        "trust_safety": 36,
        "urgency": 91,
        "confusion_risk": 65,
        "calm_aesthetic_style": 22,
        "theme_fit": 63,
        "watch_time_score": 58,
        "save_rate_score": 34,
        "click_rate_score": 76,
        "purchase_rate_score": 55,
        "platform_fit": 75,
    },
    {
        "campaign_id": "hist_004",
        "campaign_name": "Trust-building testimonial video",
        "best_segment": "Parents / guardians",
        "platform": "Facebook/YouTube",
        "visual_attention": 58,
        "emotional_warmth": 84,
        "motivation_reward": 61,
        "cognitive_load": 24,
        "memory_encoding": 74,
        "trust_safety": 88,
        "urgency": 25,
        "confusion_risk": 18,
        "calm_aesthetic_style": 86,
        "theme_fit": 77,
        "watch_time_score": 76,
        "save_rate_score": 66,
        "click_rate_score": 58,
        "purchase_rate_score": 67,
        "platform_fit": 78,
    },
    {
        "campaign_id": "hist_005",
        "campaign_name": "Cluttered feature dump",
        "best_segment": "General audience",
        "platform": "YouTube",
        "visual_attention": 52,
        "emotional_warmth": 38,
        "motivation_reward": 40,
        "cognitive_load": 85,
        "memory_encoding": 45,
        "trust_safety": 32,
        "urgency": 48,
        "confusion_risk": 88,
        "calm_aesthetic_style": 18,
        "theme_fit": 40,
        "watch_time_score": 32,
        "save_rate_score": 18,
        "click_rate_score": 24,
        "purchase_rate_score": 15,
        "platform_fit": 45,
    },
])

display(historical_campaigns)

## Cell 15 — CRM ranking logic

In [ ]:
def vector_from_dict(d, names=FEATURE_NAMES):
    return np.array([float(d[k]) for k in names], dtype=np.float32)

def cosine_similarity_0_100(a, b, eps=1e-8):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    sim = float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps))
    return round(((sim + 1) / 2) * 100, 2)

def inverse_distance_similarity_0_100(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    avg_abs_diff = float(np.mean(np.abs(a - b)))
    return round(max(0, 100 - avg_abs_diff), 2)

def crm_rank_audiences(marketing_vector):
    new_vec = vector_from_dict(marketing_vector)
    rows = []

    for _, row in historical_campaigns.iterrows():
        old_vec = row[FEATURE_NAMES].values.astype(np.float32)

        cosine_sim = cosine_similarity_0_100(new_vec, old_vec)
        distance_sim = inverse_distance_similarity_0_100(new_vec, old_vec)

        similarity = round(0.55 * cosine_sim + 0.45 * distance_sim, 2)

        engagement_score = round(
            0.45 * row["watch_time_score"] +
            0.30 * row["save_rate_score"] +
            0.25 * row["click_rate_score"],
            2,
        )

        final_score = round(
            0.40 * similarity +
            0.25 * row["purchase_rate_score"] +
            0.20 * engagement_score +
            0.15 * row["platform_fit"],
            2,
        )

        rows.append({
            "campaign_id": row["campaign_id"],
            "matched_campaign": row["campaign_name"],
            "recommended_segment": row["best_segment"],
            "platform": row["platform"],
            "similarity": similarity,
            "purchase_score": row["purchase_rate_score"],
            "engagement_score": engagement_score,
            "platform_fit": row["platform_fit"],
            "final_audience_score": final_score,
        })

    rankings = pd.DataFrame(rows).sort_values("final_audience_score", ascending=False)
    return rankings

## Cell 16 — Report builder

In [ ]:
def intent_gap_score(target, marketing_vector, emotion_result):
    target = (target or "").lower().strip()
    
    # Define ideal profiles for target emotions
    ideal_profiles = {
        "urgency": {"emotional_arousal": 0.9, "attention_focus": 0.9, "cognitive_load": 0.6},
        "trust": {"emotional_arousal": 0.4, "attention_focus": 0.8, "cognitive_load": 0.3},
        "motivation": {"emotional_arousal": 0.8, "attention_focus": 0.8, "cognitive_load": 0.5},
        "calm focus": {"emotional_arousal": 0.2, "attention_focus": 0.9, "cognitive_load": 0.2},
    }
    
    profile = ideal_profiles.get(target)
    if not profile:
        # Fallback to old heuristic if unknown target
        detected = (emotion_result.get("detected_emotion") or "").lower().strip()
        if target == detected: return 0
        if "overload" in detected: return 80
        return 50
        
    # Calculate distance between ideal and actual vector
    gap = 0
    weights = {"emotional_arousal": 40, "attention_focus": 30, "cognitive_load": 30}
    
    for metric, weight in weights.items():
        actual = marketing_vector.get(metric, 0)
        ideal = profile.get(metric, 0)
        gap += abs(ideal - actual) * weight
        
    return round(min(gap, 100))

def generate_fixes(emotion_result, marketing_vector):
    detected = emotion_result.get("detected_emotion", "Unknown")
    chaos = emotion_result.get("chaos_spike_second", "N/A")
    
    fixes = []
    
    # 1. Evaluate Cognitive Load dynamically
    cog_load = marketing_vector.get("cognitive_load", 0)
    if cog_load > 0.75:
        fixes.append(f"High cognitive load ({cog_load:.2f}) detected. Simplify text and visual clutter, especially around the complexity spike at t={chaos}s.")
    elif cog_load < 0.3:
        fixes.append(f"Low cognitive load ({cog_load:.2f}). You have room to introduce more complex messaging or specific product features without losing the viewer.")
        
    # 2. Evaluate Attention Focus dynamically
    attention = marketing_vector.get("attention_focus", 0)
    if attention < 0.45:
        fixes.append(f"Attention is dropping ({attention:.2f}). Introduce a strong visual pattern interrupt or sound effect in the first 3 seconds.")
    elif attention > 0.75:
        fixes.append(f"Strong attention maintained ({attention:.2f}). Capitalize on this by placing your primary Call-to-Action (CTA) closer to the {chaos}s mark.")
        
    # 3. Evaluate Memory Encoding dynamically
    memory = marketing_vector.get("memory_encoding", 0)
    if memory < 0.5:
        fixes.append(f"Weak memory encoding ({memory:.2f}). Ensure your brand logo and core message are held on screen longer and stand out against the background.")
        
    # 4. Evaluate Emotional Arousal dynamically
    arousal = marketing_vector.get("emotional_arousal", 0)
    if detected == "Apathy" or arousal < 0.35:
        fixes.append(f"Apathy detected (Arousal: {arousal:.2f}). The creative is too flat. Inject more vibrant colors, faster pacing, or high-energy music to wake the brain up.")
    elif "Overload" in detected or arousal > 0.85:
        fixes.append(f"Overwhelming arousal ({arousal:.2f}). The asset is bordering on chaotic; slow the pacing and focus on a single narrative thread.")
        
    # Fallback if somehow no thresholds hit
    if not fixes:
        fixes.append("The asset shows balanced neural responses. A/B test a shorter variant to see if engagement holds.")
        
    return fixes

def build_neuromark_report(
    asset_path,
    tribe_input_path,
    preds,
    segments,
    neural_stats,
    marketing_vector,
    emotion_result,
    crm_rankings,
    target_emotion: str,
    target_audience: str,
    objective: str,
):
    import numpy as np
    top_recommendation = crm_rankings.iloc[0].to_dict()

    report = {
        "schema_version": "neuromark_real_tribe_api_v2",
        "status": "success",

        "campaign": {
            "asset_name": asset_path.name,
            "tribe_input_name": tribe_input_path.name,
            "target_audience": target_audience,
            "objective": objective,
            "target_emotion": target_emotion,
        },

        "real_tribe_evidence": {
            "preds_shape": list(np.asarray(preds).shape),
            "num_timesteps": int(np.asarray(preds).shape[0]),
            "num_vertices": int(np.asarray(preds).shape[1]),
            "num_segments": len(segments),
            "neural_stats": neural_stats,
        },

        "neuromark_marketing_vector": marketing_vector,
        "emotion_measurement": emotion_result,

        "intent_vs_actual_gap": intent_gap_score(
            target_emotion,
            marketing_vector,
            emotion_result
        ),

        "crm_matching": {
            "top_recommendation": top_recommendation,
            "all_rankings": crm_rankings.to_dict(orient="records"),
        },

        "recommended_creative_fixes": generate_fixes(
            emotion_result,
            marketing_vector,
        ),

        "safe_claim": (
            "This system uses real TRIBE v2 predicted cortical activity as an input signal, "
            "then applies a transparent NeuroMark interpretation layer to produce campaign vectors, "
            "emotion categories, CRM audience recommendations, and a visual brain-response plot. "
            "It is a working prototype, not a clinically validated emotion detector or sales guarantee."
        ),
    }

    return report


## Cell 17 — Main analyzer with brain plot

In [ ]:
def analyze_asset_with_real_tribe_and_plot(
    asset_path: Path,
    target_emotion: str = "Urgency",
    target_audience: str = "Students preparing for exams",
    objective: str = "Get students to start a mock exam countdown",
    use_short_clip: bool = True,
    short_seconds: int = 5,
    return_brain_plot: bool = True,
):
    start = time.time()
    asset_id = stable_asset_id(asset_path)

    preds, segments, df_events, tribe_input_path = run_real_tribe_on_asset(
        asset_path,
        use_short_clip=use_short_clip,
        short_seconds=short_seconds,
    )

    neural_curves, neural_stats = extract_tribe_neural_dynamics(preds)
    marketing_vector = neuromark_vector_from_real_tribe(neural_curves)
    emotion_result = emotion_tree_classify(marketing_vector, neural_curves, fps=1)
    crm_rankings = crm_rank_audiences(marketing_vector)

    report = build_neuromark_report(
        asset_path=asset_path,
        tribe_input_path=tribe_input_path,
        preds=preds,
        segments=segments,
        neural_stats=neural_stats,
        marketing_vector=marketing_vector,
        emotion_result=emotion_result,
        crm_rankings=crm_rankings,
        target_emotion=target_emotion,
        target_audience=target_audience,
        objective=objective,
    )

    report["runtime_seconds"] = round(time.time() - start, 2)

    report_path = OUTPUT_DIR / f"{asset_path.stem}_{asset_id}_report.json"
    vector_path = OUTPUT_DIR / f"{asset_path.stem}_{asset_id}_vector.csv"
    crm_path = OUTPUT_DIR / f"{asset_path.stem}_{asset_id}_crm.csv"

    pd.DataFrame([marketing_vector]).to_csv(vector_path, index=False)
    crm_rankings.to_csv(crm_path, index=False)

    saved_files = {
        "report_json": str(report_path),
        "marketing_vector_csv": str(vector_path),
        "crm_rankings_csv": str(crm_path),
    }

    if return_brain_plot:
        brain_plot_path, brain_plot_b64 = make_tribe_brain_plot_base64(
            preds=preds,
            segments=segments,
            asset_id=asset_id,
            max_timesteps=15,
        )

        report["brain_plot"] = {
            "filename": brain_plot_path.name,
            "mime_type": "image/png",
            "base64": brain_plot_b64,
        }

        saved_files["brain_plot_png"] = str(brain_plot_path)

    report["saved_files"] = saved_files

    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    return report

## Cell 18 — Optional local test with your known Drive video

In [ ]:
TEST_WITH_KNOWN_DRIVE_VIDEO = False

if TEST_WITH_KNOWN_DRIVE_VIDEO:
    test_video_path = UPLOAD_DIR / "fastpapers_mockexam_countdown.mp4"

    !gdown "https://drive.google.com/uc?id=1W88VUGFXejqngaXNU1DaiKCTzjF6UkL3" -O "{test_video_path}"

    test_report = analyze_asset_with_real_tribe_and_plot(
        asset_path=test_video_path,
        target_emotion="Urgency",
        target_audience="Students preparing for exams",
        objective="Get students to start a mock exam countdown",
        use_short_clip=True,
        short_seconds=5,
        return_brain_plot=True,
    )

    print("Report keys:", test_report.keys())
    print("Has brain_plot:", "brain_plot" in test_report)
    print(json.dumps({k: v for k, v in test_report.items() if k != "brain_plot"}, indent=2))

## Cell 19 — FastAPI app

In [ ]:
app = FastAPI(title="NeuroMark Real TRIBE Research API with Brain Plot")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

REQUEST_LOGS = []
JOBS = {}
import uuid
import threading

@app.get("/")
def root():
    return {
        "app": "NeuroMark Real TRIBE Research API with Brain Plot",
        "status": "running",
        "endpoints": ["/health", "/analyze", "/analyze_with_plot", "/logs"],
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

@app.get("/logs")
def logs():
    return REQUEST_LOGS[-20:]

@app.delete("/logs")
def clear_logs():
    REQUEST_LOGS.clear()
    return {"status": "cleared"}

## Cell 20 — API endpoints

In [ ]:
@app.post("/analyze")
async def analyze(
    request: Request,
    file: UploadFile = File(...),
    target_emotion: str = Form("Urgency"),
    target_audience: str = Form("Students preparing for exams"),
    objective: str = Form("Get students to start a mock exam countdown"),
    use_short_clip: bool = Form(True),
    short_seconds: int = Form(5),
):
    try:
        filename = safe_filename(file.filename)
        save_path = UPLOAD_DIR / filename

        with open(save_path, "wb") as f:
            shutil.copyfileobj(file.file, f)

        if not (is_video(save_path) or is_image(save_path)):
            return JSONResponse(
                status_code=400,
                content={
                    "status": "error",
                    "message": "Unsupported file type. Use MP4, JPG, JPEG, or PNG.",
                },
            )

        report = analyze_asset_with_real_tribe_and_plot(
            asset_path=save_path,
            target_emotion=target_emotion,
            target_audience=target_audience,
            objective=objective,
            use_short_clip=use_short_clip,
            short_seconds=short_seconds,
            return_brain_plot=False,
        )

        REQUEST_LOGS.append({
            "time": time.time(),
            "filename": filename,
            "status": "success",
            "runtime_seconds": report.get("runtime_seconds"),
            "has_brain_plot": False,
        })

        return report

    except Exception as e:
        REQUEST_LOGS.append({
            "time": time.time(),
            "filename": getattr(file, "filename", None),
            "status": "error",
            "error": str(e),
        })

        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})


@app.post("/analyze_with_plot")
async def analyze_with_plot(
    request: Request,
    file: UploadFile = File(...),
    target_emotion: str = Form("Urgency"),
    target_audience: str = Form("Students preparing for exams"),
    objective: str = Form("Get students to start a mock exam countdown"),
    use_short_clip: bool = Form(True),
    short_seconds: int = Form(5),
    return_brain_plot: bool = Form(True),
):
    try:
        filename = safe_filename(file.filename)
        save_path = UPLOAD_DIR / filename

        with open(save_path, "wb") as f:
            shutil.copyfileobj(file.file, f)

        if not (is_video(save_path) or is_image(save_path)):
            return JSONResponse(
                status_code=400,
                content={
                    "status": "error",
                    "message": "Unsupported file type. Use MP4, JPG, JPEG, or PNG.",
                },
            )

        report = analyze_asset_with_real_tribe_and_plot(
            asset_path=save_path,
            target_emotion=target_emotion,
            target_audience=target_audience,
            objective=objective,
            use_short_clip=use_short_clip,
            short_seconds=short_seconds,
            return_brain_plot=return_brain_plot,
        )

        REQUEST_LOGS.append({
            "time": time.time(),
            "filename": filename,
            "status": "success_with_brain_plot",
            "runtime_seconds": report.get("runtime_seconds"),
            "has_brain_plot": bool(report.get("brain_plot")),
        })

        return report

    except Exception as e:
        REQUEST_LOGS.append({
            "time": time.time(),
            "filename": getattr(file, "filename", None),
            "status": "error",
            "error": str(e),
        })

        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})
def background_process(job_id, save_path, target_emotion, target_audience, objective, use_short_clip, short_seconds, return_brain_plot):
    try:
        JOBS[job_id]["progress"] = "Analyzing with TRIBE v2 (this takes 60-120s)..."
        report = analyze_asset_with_real_tribe_and_plot(
            asset_path=save_path,
            target_emotion=target_emotion,
            target_audience=target_audience,
            objective=objective,
            use_short_clip=use_short_clip,
            short_seconds=short_seconds,
            return_brain_plot=return_brain_plot,
        )
        JOBS[job_id]["status"] = "completed"
        JOBS[job_id]["result"] = report
        JOBS[job_id]["progress"] = "Analysis complete."
    except Exception as e:
        JOBS[job_id]["status"] = "failed"
        JOBS[job_id]["error"] = str(e)
        JOBS[job_id]["progress"] = "Failed."

@app.post("/submit_job")
async def submit_job(
    request: Request,
    file: UploadFile = File(...),
    target_emotion: str = Form("Urgency"),
    target_audience: str = Form("Students preparing for exams"),
    objective: str = Form("Get students to start a mock exam countdown"),
    use_short_clip: bool = Form(True),
    short_seconds: int = Form(5),
    return_brain_plot: bool = Form(True),
):
    try:
        job_id = str(uuid.uuid4())
        filename = safe_filename(file.filename)
        save_path = UPLOAD_DIR / f"{job_id}_{filename}"
        with open(save_path, "wb") as f:
            shutil.copyfileobj(file.file, f)
        JOBS[job_id] = {"status": "processing", "progress": "Upload received. Booting GPU...", "result": None, "error": None}
        threading.Thread(target=background_process, args=(job_id, save_path, target_emotion, target_audience, objective, use_short_clip, short_seconds, return_brain_plot)).start()
        return {"status": "success", "job_id": job_id}
    except Exception as e:
        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

@app.get("/job_status/{job_id}")
def job_status(job_id: str):
    if job_id not in JOBS:
        return JSONResponse(status_code=404, content={"status": "error", "message": "Job not found"})
    return JOBS[job_id]


## Cell 21 — Start local Colab server

In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

print("Server started on port 8000 inside Colab.")

## Cell 22 — Test local health endpoint

In [ ]:
health_response = requests.get("http://127.0.0.1:8000/health").json()
health_response

## Cell 23 — Start ngrok tunnel

In [ ]:
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_TOKEN"])

public_url = ngrok.connect(8000).public_url

print("Public API URL:")
print(public_url)

print("\nFrontend should POST to:")
print(f"{public_url}/analyze_with_plot")

print("\nHealth check:")
print(f"{public_url}/health")

## Cell 24 — Optional public API test

In [ ]:
RUN_PUBLIC_TEST = False

if RUN_PUBLIC_TEST:
    test_video_path = UPLOAD_DIR / "fastpapers_mockexam_countdown.mp4"
    if not test_video_path.exists():
        !gdown "https://drive.google.com/uc?id=1W88VUGFXejqngaXNU1DaiKCTzjF6UkL3" -O "{test_video_path}"

    files_payload = {"file": open(test_video_path, "rb")}
    data_payload = {
        "target_emotion": "Urgency",
        "target_audience": "Students preparing for exams",
        "objective": "Get students to start a mock exam countdown",
        "use_short_clip": "true",
        "short_seconds": "5",
        "return_brain_plot": "true",
    }

    response = requests.post(
        f"{public_url}/analyze_with_plot",
        files=files_payload,
        data=data_payload,
    )

    print(response.status_code)
    result = response.json()
    print("keys:", result.keys())
    print("has brain_plot:", "brain_plot" in result)
    print(json.dumps({k: v for k, v in result.items() if k != "brain_plot"}, indent=2))

## Cell 25 — Save outputs as ZIP

In [ ]:
zip_path = BASE_DIR / "neuromark_api_outputs.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob("*"):
        z.write(p, arcname=p.name)

print("Created:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

from google.colab import files
files.download(str(zip_path))